Regular expression to filter :

Anchors: "^patter$" <br>
^ - Start of String <br>
$ - end of string <br>



1. rlike()
2. regexp_replace()
3. regexp_extract()


In [0]:
import pyspark.sql.functions as F

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType
from pyspark.sql.functions import lit

spark = SparkSession.builder.getOrCreate()

# Create sample data with various regex patterns
data = [
    ("john@example.com", "1234567890", "2023-12-01", "https://example.com", "INFO: User logged in", "A1B-2C3", "192.168.1.1", "VISA 4111-1111-1111-1111", "Special $100 bonus!"),
    ("invalid.email", "5551234", "12/31/23", "www.site.com", "ERROR: File not found", "XY-9Z8", "256.300.400.500", "MC 5500-0000-0000-0004", "Price: €50"),
    ("jane.doe@co.uk", "(987) 654-3210", "2023Q4", "http://test.org/path", "WARN: Low memory", "3D4E-5F6", "10.0.0.1", "AMEX 3782-822463-10005", "Code: #ABCD123"),
    ("missing@domain", "1-800-HELP", "20230515", "invalid.url", "DEBUG: Session ID=a1b2c3", "NO-DASH", "172.16.254.1", "INVALID 1234-5678", "Remove_underscores!"),
    ("test@sub.domain.com", "+44 20 7946 0958", "Feb 28 2024", "https://secure.site?q=1", "FATAL: System crash", "PROD-001", "2001:0db8:85a3:0000:0000:8a2e:0370:7334", "DISC 6011-0001-0000-0002", "Mix123 of! characters")
]

columns = [
    "emails", "phone_numbers", "dates", "urls", "log_entries",
    "product_codes", "ip_addresses", "credit_cards", "free_text"
]

df = spark.createDataFrame(data, schema=columns)
df = df.withColumn("additional_text", lit(
    """Contact: support@company.com, 
    IP: 10.0.0.42, 
    Code: ABC-XYZ-123, 
    Mixed@$ symbols123"""
))

df.show(truncate=True)

In [0]:
%run /PySpark/Basics/dataframes

In [0]:
import re

In [0]:
print(df.columns)
print(df.count())

In [0]:
df.printSchema()

### rlike()

In [0]:
# Filter rows that contains only numeric value
from pyspark.sql.functions import col
df.filter(col("phone_numbers").rlike("^[0-9]*$")).show()

In [0]:
# how to filter valid emails xyz@abc.pqr
email_expression = "^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9._%+-].[a-zA-Z]{2,}$"
df.filter(col("emails").rlike(email_expression)).show()


In [0]:
df.show(truncate=True)

In [0]:
# validate URL
url_expression = "^https://"
df.filter(col("urls").rlike(url_expression)).show()

In [0]:
# ip address validation 
# 111.111.888.900
ip_expression = "^((25[0-5]|2[0-4][0-9]|1[0-9]{2}|[1-9]?[0-9])\.){3}(25[0-5]|2[0-4][0-9]|1[0-9]{2}|[1-9]?[0-9])$"
df.filter(col("ip_addresses").rlike(ip_expression)).show()

### regexp_replace 
This function is used to replace substrings in a column using regular expression

In [0]:
data = [(1, "Rohan-Srivastwa"),\
        (2, "Priyanshu-Baneerjee"),\
        (3, "Avinash kumar"),\
        (10, "Sachin maggu")]
schema = ["id", "name"]
df =  spark.createDataFrame(data=data, schema=schema)
df.show()

In [0]:
from pyspark.sql.functions import regexp_replace
df.select(col("*")).withColumn("new_name", regexp_replace(col("name"),"-"," ")).show()

In [0]:
# split() function
from pyspark.sql.functions import regexp_replace, split
df.select(col("*")).withColumn("new_name", split(regexp_replace(col("name"),"-"," "), " ")).show()

In [0]:
# split() function
from pyspark.sql.functions import regexp_replace, split
df.select(col("*")).withColumn("new_name", split(regexp_replace(col("name"),"-"," "), " "))\
  .withColumn("first_name", col("new_name")[0])\
  .withColumn("last_name", col("new_name")[1])\
  .drop("name","new_name")\
  .show()

### regexp_extract